##مقدمة صغيرة.كانت إحدى مهامي في وظيفتي السابقة هي توفير المعدات الطبية للمستشفيات في روسيا. <br/>
في هذه اللحظة هناك لائحة حكومية بشأن الخيارات الممكنة لشراء المعدات الطبية في المستشفيات. <br/>
يستدعي الإجراء "المناقصة" ويتم وضع المناقصات على الموقع العام. يمكن للجميع التقدم للمشاركة والفوز بالمناقصة. الشخص الذي يقدم أقل سعر سيكون هو الفائز.
في عالم مثالي، لا ينبغي للمواصفات الفنية للمعدات الضرورية أن تشير إلى علامة تجارية محددة للشركة المصنعة. (كمثال - تشبيه: العميل لا يستطيع الكتابة - نريد شراء "أحدث طراز من iMac." سيكون من الصحيح الكتابة: نحتاج إلى جهاز كمبيوتر مزود بـ 8 مراكز، بتردد 2.2 ميجاهرتز، وذاكرة وصول عشوائي (RAM) سعة 8 جيجابايت. وقرص صلب لا يقل عن 250 جيجابايت، وما إلى ذلك)<br/>
ولكن في الواقع، العديد من الشركات المصنعة لديها مجموعة خاصة بها من الخصائص التقنية الفريدة التي تحدد الشركة بشكل فريد. وتم تصميم المهمة الفنية للمناقصة بحيث تتوافق علامة تجارية واحدة فقط مع الوصف.<br/>
كان ذلك في حالتي - المعدات الطبية.<br/>
أنا، كمورد، مهتم جدًا بالتعرف بسرعة على شركة تصنيع معينة حتى أكون أول من يتفاوض ويحصل على الحد الأدنى من السعر. وهذا ضمان الفوز في العطاء.<br/>
لقد قمت بتجميع محلل يسحب البيانات من الموقع (zakupki.gov.ru)، حيث يتم نشر نتائج المناقصات العامة. هنا، بالمناسبة، هناك عدة خيارات ممكنة. قم بالتحليل من الصفحة، أو من ftp، أو احصل على json من الأشخاص الذين يشاركون المعلومات المميزة بالفعل (Proект КГИ “Госзатраты” (https://clearspending.ru)<br/>
أخذت البيانات لعام 2017. وفي منطقة سفيردلوفسك (الكيان الجغرافي في روسيا).
النتائج عبارة عن عقد يحتوي على عدد كبير من التفاصيل. منذ البدايةنعم، لم يكن من المخطط استخدام ML. لقد قمت بجمع البيانات من أجل تحليلات لمرة واحدة فقط. تم تفريغ المؤشرات التالية:
- customer.inn - المعرف الفريد للعميل (المستشفى)
- regNum - المعرف الفريد للعقد
- تاريخ التوقيع - تاريخ توقيع العقد
- الاسم - المواصفات الفعلية لتسليم المعدات المقترحة.
- سعر المنتج - سعر المعدات
- الكمية - كمية المعدات
- نزل - المعرف الفريد للفائز
- الصانع - الهدف . اسماء الشركات المصنعة
رابط البيانات - https://drive.google.com/open?id=1S9X_B9Vayev_mu9co8mVR0acdeU5uBTw <br/>
هذه المهمة مشابهة للمنافسة المتوسطة. <br/>
بدلا من محتوى المقالات - تلك المهمة. <br/>
وميزات مختلفة لتحسين ممكن في السرعة. <br/>


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from scipy.sparse import csr_matrix
from scipy.sparse import hstack
from sklearn.metrics import confusion_matrix

In [ ]:
data = pd.read_csv('Urology_department.csv', delimiter=';', converters={'customer.inn':str,\
                                                                                                           'signDate':pd.to_datetime})
data.head()

In [ ]:
data.info()

In [ ]:
data.isnull().any()

In [ ]:
data = data.dropna()


حسنًا. لا توجد قيم NAN. جميع الأنواع صحيحة


In [ ]:
data.shape


دعونا نلقي نظرة على المتغير المستهدف لدينا


In [ ]:
data.Manufacturer.value_counts()


هناك العديد من القيم "0". هذا يعني أننا لا نعرف الشركة المصنعة. سنقوم بإسقاط هذه الصفوف. وقامت إحدى الشركات المصنعة بتكرار ("Coloplast" و"COLOPLAST"). سنقوم بتقليل أسماء الشركات المصنعة المستهدفة. ونحن لا نهتم بالأعداد الأقل من 10. (وهي نادرة جدًا بحيث يمكن إهمالها).
بالمناسبة - قضيت معظم الوقت في إعداد البيانات. لأن العقود المنشورة لا تحتوي على معلومات عن الشركات المصنعة. إما أنه كذلك، لكنه غير منظم للغاية. لقد قمت بإنشاء متغير الهدف بنفسي.


In [ ]:
data.Manufacturer = data.Manufacturer.str.lower()
cnt = data.Manufacturer.value_counts()
data = data.loc[(data.Manufacturer != '0') & (data.Manufacturer.isin(cnt.index[cnt >= 10]).values)]

In [ ]:
plt.figure(figsize=(20,10))
sns.countplot(y="Manufacturer", data=data, order = data.Manufacturer.value_counts().index)


مهمتي هي إنشاء مصنف لـ 14 فئة



دعونا نلقي نظرة على المتغيرات الفئوية الأخرى. Coster.inn ونزل الفائز


In [ ]:
plt.figure(figsize=(20,10))
sns.countplot(y="customer.inn", data=data, order = data["customer.inn"].value_counts().index)


والفائز. نزل


In [ ]:
plt.figure(figsize=(20,10))
sns.countplot(y="inn", data=data, order = data["inn"].value_counts().index)


لدينا قادة من حيث المشتريات والإمدادات. <br/>
دعونا نلقي نظرة على العلامات التجارية التي يبيعها المصنعون للمستشفيات


In [ ]:
cnt_l_c = data["customer.inn"].value_counts()
df_leaders_costumers = data.loc[data["customer.inn"].isin(cnt_l_c.index[cnt_l_c >= 100]).values]
plt.figure(figsize=(20,10))
sns.countplot(y="Manufacturer", data=df_leaders_costumers, order = df_leaders_costumers.Manufacturer.value_counts().index)

لا أعتقد أن هذه الميزة ستكون مفيدة..


In [ ]:
cnt_l_s = data["inn"].value_counts()
df_leaders_suppliers = data.loc[data["inn"].isin(cnt_l_s.index[cnt_l_s >= 100]).values]
plt.figure(figsize=(20,10))
sns.countplot(y="Manufacturer", data=df_leaders_suppliers, order = df_leaders_suppliers.Manufacturer.value_counts().index)


يمكننا أن نرى أنه لم يتم عرض بعض الشركات المصنعة المستهدفة. لكن التوزيع يبدو متشابهًا في بدايتنا. 



ودعونا ننظر إلى تاريخ العقد


In [ ]:
data['day'] = data['signDate'].apply(pd.datetime.weekday)
data['month'] = data['signDate'].apply(lambda x: x.month)

In [ ]:
plt.figure(figsize=(20,10))
sns.countplot(y="Manufacturer", data=data, hue='month')

In [ ]:
plt.figure(figsize=(20,10))
sns.countplot(y="Manufacturer", data=data, hue='day')


في المرة الأولى، لنقم ببناء نموذج يحتوي فقط على وصف الشركة المصنعة كميزة


In [ ]:
train_part, test_part = train_test_split(data[['name','Manufacturer']], test_size=0.2, random_state=21, stratify=data["Manufacturer"])

In [ ]:
pipeline_tfidf_lr = Pipeline([('tfidf', TfidfVectorizer()),
                              ('lr', LogisticRegression())])

pipeline_tfidf_lr.fit(train_part['name'], train_part["Manufacturer"])

predicted = pipeline_tfidf_lr.predict(test_part["name"])


مصفوفة الارتباك


In [ ]:
test_classes_counts = test_part["Manufacturer"].value_counts()
test_classes_names = np.array(test_classes_counts.index)
total_classes = len(test_classes_counts)


cm = confusion_matrix(y_true=test_part["Manufacturer"], y_pred=predicted, labels=test_classes_names)
for true_class_id in range(total_classes):
    true_class_name = test_classes_names[true_class_id]
    true_class_count = test_classes_counts[true_class_name]
    
    print('For Manufacturer "{0}" ({1} test examples) were predicted:'.format(true_class_name, true_class_count))
    for pred_class_id in range(total_classes):
        percent = int(cm[true_class_id, pred_class_id].item()) / int(true_class_count.item()) * 100
        if percent >= 5:
            pred_class_name = test_classes_names[pred_class_id]
            print('\t"{0}" в {1:.2f} % ({2} раз)'.format(pred_class_name, percent, cm[true_class_id, pred_class_id]))

In [ ]:
time_split = TimeSeriesSplit(n_splits=5)

In [ ]:
from sklearn.model_selection import GridSearchCV

parameters_lr = {'tfidf__ngram_range': [(1, 1), (1, 2)],
                 'tfidf__use_idf': (True, False),
                 'tfidf__max_features': [50000, 100000],
                 'lr__C': np.logspace(-2, 2, 10),
                 }

gs_lr = GridSearchCV(pipeline_tfidf_lr, parameters_lr, scoring="accuracy", n_jobs=4, cv=time_split, verbose=10,
                     return_train_score=True)
gs_lr = gs_lr.fit(data["name"], data["Manufacturer"])

In [ ]:
gs_lr.best_params_


دعونا نحاول إعداد البيانات النصية مع إزالة الأحرف الخاصة. <br/>
ووقف ذلك


In [ ]:
import nltk
from nltk.stem import  SnowballStemmer
import re
stemmer = SnowballStemmer('russian')

In [ ]:
def remove_spec_char(string):
    return re.sub('[?|#|$|.|!|0-9|²|)|(|,|–|+|”|—|’|/]', '', string)

def steming(string):
    singles = [stemmer.stem(word) for word in string.split()]
    return " ".join(singles)

In [ ]:
data.name = data.name.apply(remove_spec_char)
data.name = data.name.apply(steming)


تدريب نموذجنا على البيانات المعدة


In [ ]:
pipeline_tfidf_lr_prep = Pipeline(
    [('tfidf', TfidfVectorizer(ngram_range=(1, 2), use_idf=True, max_features=50000)),
     ('lr', LogisticRegression(C=12.915496650148826)),
     ])

In [ ]:
train_part, test_part = train_test_split(data[['name','Manufacturer']], test_size=0.2, random_state=21, stratify=data["Manufacturer"])
pipeline_tfidf_lr_prep.fit(train_part["name"], train_part["Manufacturer"])
predicted = pipeline_tfidf_lr_prep.predict(test_part["name"])

In [ ]:
test_classes_counts = test_part["Manufacturer"].value_counts()
test_classes_names = np.array(test_classes_counts.index)
total_classes = len(test_classes_counts)

cm = confusion_matrix(y_true=test_part["Manufacturer"], y_pred=predicted, labels=test_classes_names)
for true_class_id in range(total_classes):
    true_class_name = test_classes_names[true_class_id]
    true_class_count = test_classes_counts[true_class_name]
    
    print('For Manufacturer "{0}" ({1} test examples) were predicted:'.format(true_class_name, true_class_count))
    for pred_class_id in range(total_classes):
        percent = int(cm[true_class_id, pred_class_id].item()) / int(true_class_count.item()) * 100
        if percent >= 5:
            pred_class_name = test_classes_names[pred_class_id]
            print('\t"{0}" в {1:.2f} % ({2} раз)'.format(pred_class_name, percent, cm[true_class_id, pred_class_id]))


الآن دعونا نضيف ميزات أخرى، ونلقي نظرة على النتائج


In [ ]:
scaler = StandardScaler()
tfidf = TfidfVectorizer(ngram_range=(1, 2), use_idf=True, max_features=50000)
lr = LogisticRegression(C=12.915496650148826)

In [ ]:
data = pd.get_dummies(data, columns=['customer.inn', 'inn', 'day', 'month'])
data.drop(columns=['regNum','signDate'], inplace=True)

In [ ]:
tmp = StandardScaler().fit_transform(data[['product_price','quantity']])
text = tfidf.fit_transform(data.name)
features = data.iloc[:,4:].values

In [ ]:
X = csr_matrix(hstack([text, tmp, features]))
y = data.Manufacturer
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21, stratify=y)

In [ ]:
lr.fit(X_train, y_train)
predicted = lr.predict(X_test)

In [ ]:
test_classes_counts = test_part["Manufacturer"].value_counts()
test_classes_names = np.array(test_classes_counts.index)
total_classes = len(test_classes_counts)

In [ ]:
cm = confusion_matrix(y_true=test_part["Manufacturer"], y_pred=predicted, labels=test_classes_names)

In [ ]:
for true_class_id in range(total_classes):
    true_class_name = test_classes_names[true_class_id]
    true_class_count = test_classes_counts[true_class_name]
    
    print('For Manufacturer "{0}" ({1} test examples) were predicted:'.format(true_class_name, true_class_count))
    for pred_class_id in range(total_classes):
        percent = int(cm[true_class_id, pred_class_id].item()) / int(true_class_count.item()) * 100
        if percent >= 5:
            pred_class_name = test_classes_names[pred_class_id]
            print('\t"{0}" в {1:.2f} % ({2} раз)'.format(pred_class_name, percent, cm[true_class_id, pred_class_id]))


لذا، فإننا نزيد درجاتنا بميزات جديدة



### السبب الرئيسي لاستكمال هذه المهمة هو:
- تقليل الوقت اللازم لمعالجة المناقصات الواردة <br/>
- الحصول على شروط مربحة من الشركات المصنعة (تذكر من هو أول من يطلب - من لديه الحد الأقصى للخصم)<br/>
- وهذا هو مفتاح الفوز بالمناقصة.<br/>
- تقليص أو التحول إلى مهام أخرى للموظفين المؤهلين في المنظمة.<br/>
- خفض التكاليف وتنظيم نمو الأرباح.



### ما الذي يمكن فعله بعد ذلك:
- الاتجاه الرئيسي الذي سأختاره هو الإثراء بالبيانات الجديدة. الاستيلاء على عقود جديدة، ووضع علامات على الشركات المصنعة غير المعروفة.
- استخدام النماذج الأخرى (الغابات العشوائية وsvm وغيرها...)